#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn

from pathlib import Path
from scipy import sparse

from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import lightgbm as lgb

#### Loading Processed Datasets and Feature Matrices

In [2]:
processed_dir = Path("../data/processed")
features_dir = Path("../data/features")
models_dir = Path("../models")

train_df = pd.read_csv(processed_dir / "train.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

y_train = train_df["intent"]
y_test = test_df["intent"]

X_train_word = sparse.load_npz(features_dir / "X_train_word.npz")
X_test_word = sparse.load_npz(features_dir / "X_test_word.npz")

X_train_char = sparse.load_npz(features_dir / "X_train_char.npz")
X_test_char = sparse.load_npz(features_dir / "X_test_char.npz")

X_train_combined = sparse.load_npz(features_dir / "X_train_combined.npz")
X_test_combined = sparse.load_npz(features_dir / "X_test_combined.npz")

print(f"Training Samples: {len(y_train):,}")
print(f"Testing Samples: {len(y_test):,}")
print(f"Number of Intent Classes: {y_train.nunique()}")

print(f"\nWord TF-IDF: {X_train_word.shape}")
print(f"Character TF-IDF: {X_train_char.shape}")
print(f"Combined TF-IDF: {X_train_combined.shape}")

print(f"\nTraining Labels Missing: {y_train.isnull().sum()}")
print(f"Testing Labels Missing: {y_test.isnull().sum()}")

Training Samples: 17,210
Testing Samples: 4,303
Number of Intent Classes: 27

Word TF-IDF: (17210, 2749)
Character TF-IDF: (17210, 6190)
Combined TF-IDF: (17210, 8939)

Training Labels Missing: 0
Testing Labels Missing: 0


#### Observations
- **17,210 training samples and 4,303 testing samples** were successfully loaded, covering all 27 customer intent classes.
- **Three TF-IDF representations are available:** word-level (2,749 features), character-level (6,190 features), and combined (8,939 features).
- **Decision:** Model development and hyperparameter tuning will use training data exclusively, preserving the held-out test set for final evaluation.

#### 2. Experiment Tracking

#### Configuring MLflow Experiment Tracking

In [3]:
mlflow_db_path = (Path.cwd().parent / "mlflow.db").resolve()

mlflow.set_tracking_uri(f"sqlite:///{mlflow_db_path.as_posix()}")

experiment_name = "Customer Support Intent Intelligence"
mlflow.set_experiment(experiment_name)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment Name: {experiment_name}")
print(f"Experiment ID: {mlflow.get_experiment_by_name(experiment_name).experiment_id}")

2026/11/04 22:39:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/04 22:39:17 INFO mlflow.store.db.utils: Updating database tables
2026/11/04 22:39:19 INFO mlflow.tracking.fluent: Experiment with name 'Customer Support Intent Intelligence' does not exist. Creating a new experiment.


MLflow Tracking URI: sqlite:///D:/Projects/ML and Data Science/Customer Support Intent Intelligence/mlflow.db
Experiment Name: Customer Support Intent Intelligence
Experiment ID: 1


#### Observations
- **MLflow experiment tracking was initialized successfully** using a project-level SQLite database.
- **The experiment `Customer Support Intent Intelligence` was created with ID 1**, providing a dedicated environment for model comparisons.
- **Decision:** Model configurations, cross-validation metrics, and relevant artifacts will be recorded in MLflow to support reproducibility and model selection.

#### Configuring Stratified Cross-Validation

In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro"
}

print(f"Cross-Validation Strategy: {cv.__class__.__name__}")
print(f"Number of Folds: {cv.n_splits}")
print(f"Shuffling Enabled: {cv.shuffle}")
print(f"Random State: {cv.random_state}")

print("\nEvaluation Metrics:")
for metric in scoring:
    print(f"- {metric}")

Cross-Validation Strategy: StratifiedKFold
Number of Folds: 5
Shuffling Enabled: True
Random State: 42

Evaluation Metrics:
- accuracy
- f1_macro
- f1_weighted
- precision_macro
- recall_macro


#### Observations
- **5-fold stratified cross-validation** is configured with shuffling and a fixed random state of 42 for reproducibility.
- **Five evaluation metrics** will be tracked, including accuracy, macro F1-score, weighted F1-score, macro precision, and macro recall.
- **Decision:** Macro F1-score will serve as the primary model-selection metric to account for performance differences across the 27 imbalanced intent classes.

#### 3. Baseline Modeling

#### Evaluating the Dummy Classifier Baseline

In [5]:
dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

dummy_cv_results = cross_validate(
    dummy_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

dummy_metrics = {
    metric: dummy_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Dummy Classifier - Baseline"):
    mlflow.log_param("model", "DummyClassifier")
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_param("feature_representation", "word_tfidf")
    mlflow.log_param("cv_folds", 5)

    for metric, value in dummy_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

dummy_results_df = pd.DataFrame(
    [dummy_metrics],
    index=["Dummy Classifier"]
).round(4)

display(dummy_results_df)

,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
Dummy Classifier,0.2029,0.0125,0.0685,0.0075,0.037


#### Observations
- **The Dummy Classifier achieves 20.29% accuracy**, reflecting the proportion of the majority intent class, `payment_issue`.
- **Macro F1-score is only 0.0125**, with macro recall of 0.0370, indicating extremely poor performance across the 27 intent classes.
- **Key Finding:** Majority-class predictions provide an inadequate classification strategy, reinforcing the importance of macro F1-score for evaluating subsequent models.

## 4. Probabilistic Model Development

#### Evaluating Multinomial Naive Bayes

In [6]:
mnb_model = MultinomialNB(alpha=1.0)

mnb_cv_results = cross_validate(
    mnb_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

mnb_metrics = {
    metric: mnb_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Multinomial Naive Bayes - Word TF-IDF"):
    mlflow.log_param("model", "MultinomialNB")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("feature_representation", "word_tfidf")
    mlflow.log_param("cv_folds", 5)

    for metric, value in mnb_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

mnb_results_df = pd.DataFrame(
    [mnb_metrics],
    index=["Multinomial Naive Bayes"]
).round(4)

display(mnb_results_df)

,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
Multinomial Naive Bayes,0.9794,0.8921,0.9767,0.9511,0.8699


## 5. Model Performance Tracking

#### Initializing the Model Comparison Results

In [7]:
model_results = []

for model_name, metrics in [
    ("Dummy Classifier", dummy_metrics),
    ("Multinomial Naive Bayes", mnb_metrics)
]:
    model_results.append({
        "Model": model_name,
        "Feature Representation": "Word TF-IDF",
        **metrics
    })

results_df = pd.DataFrame(model_results)

results_path = processed_dir / "model_comparison_results.csv"
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699



Results saved to: ..\data\processed\model_comparison_results.csv


#### Evaluating Complement Naive Bayes (Word TF-IDF)

In [8]:
cnb_model = ComplementNB(alpha=1.0)

cnb_cv_results = cross_validate(
    cnb_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

cnb_metrics = {
    metric: cnb_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Complement Naive Bayes - Word TF-IDF"):
    mlflow.log_param("model", "ComplementNB")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("feature_representation", "word_tfidf")
    mlflow.log_param("cv_folds", 5)

    for metric, value in cnb_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Complement Naive Bayes",
    "Feature Representation": "Word TF-IDF",
    **cnb_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Complement Naive Bayes achieves 98.13% accuracy** and a weighted F1-score of 0.9807, outperforming Multinomial Naive Bayes.
- **Macro F1-score improves from 0.8921 to 0.9339**, while macro recall increases from 0.8699 to 0.9177, indicating better overall performance across intent classes.
- **Key Finding:** Complement Naive Bayes is the strongest model evaluated so far, demonstrating its effectiveness for imbalanced customer intent classification.

## 6. Linear Model Development

#### Evaluating Logistic Regression (Word TF-IDF)

In [9]:
lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

lr_cv_results = cross_validate(
    lr_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

lr_metrics = {
    metric: lr_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Logistic Regression - Word TF-IDF"):
    mlflow.log_params({
        "model": "LogisticRegression",
        "C": 1.0,
        "max_iter": 1000,
        "solver": "lbfgs",
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in lr_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Logistic Regression",
    "Feature Representation": "Word TF-IDF",
    **lr_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Logistic Regression achieves 99.57% accuracy** and a weighted F1-score of 0.9956, outperforming both Naive Bayes classifiers.
- **Macro F1-score reaches 0.9859**, with macro precision of 0.9960 and macro recall of 0.9778, demonstrating strong performance across the 27 intent classes.
- **Key Finding:** Logistic Regression is currently the strongest model, establishing a high benchmark for subsequent classifiers.

#### Evaluating Linear SVM (Word TF-IDF)

In [10]:
svm_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

svm_cv_results = cross_validate(
    svm_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

svm_metrics = {
    metric: svm_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Linear SVM - Word TF-IDF"):
    mlflow.log_params({
        "model": "LinearSVC",
        "C": 1.0,
        "max_iter": 2000,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in svm_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Linear SVM",
    "Feature Representation": "Word TF-IDF",
    **svm_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Linear SVM achieves 99.75% accuracy** and a weighted F1-score of 0.9975, outperforming all previously evaluated models.
- **Macro F1-score improves to 0.9924**, with macro precision of 0.9954 and macro recall of 0.9904, indicating consistently strong classification performance across intent classes.
- **Key Finding:** Linear SVM is currently the strongest classifier, demonstrating the effectiveness of linear models on high-dimensional Word TF-IDF features.

#### Evaluating SGD Classifier (Word TF-IDF)

In [11]:
sgd_model = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=0.0001,
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

sgd_cv_results = cross_validate(
    sgd_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

sgd_metrics = {
    metric: sgd_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="SGD Classifier - Word TF-IDF"):
    mlflow.log_params({
        "model": "SGDClassifier",
        "loss": "log_loss",
        "penalty": "l2",
        "alpha": 0.0001,
        "max_iter": 1000,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in sgd_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "SGD Classifier",
    "Feature Representation": "Word TF-IDF",
    **sgd_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **SGD Classifier achieves 99.44% accuracy** and a weighted F1-score of 0.9943, demonstrating strong performance with Word TF-IDF features.
- **Macro F1-score reaches 0.9823**, slightly below Logistic Regression (0.9859) and Linear SVM (0.9924).
- **Key Finding:** Linear SVM remains the strongest classifier, while SGD Classifier provides another competitive linear modeling approach.

## 7. Tree-Based Ensemble Model Development

#### Evaluating Random Forest (Word TF-IDF)

In [12]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=1
)

rf_cv_results = cross_validate(
    rf_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=2
)

rf_metrics = {
    metric: rf_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Random Forest - Word TF-IDF"):
    mlflow.log_params({
        "model": "RandomForestClassifier",
        "n_estimators": 150,
        "max_features": "sqrt",
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in rf_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Random Forest",
    "Feature Representation": "Word TF-IDF",
    **rf_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719
6,Random Forest,Word TF-IDF,0.9935,0.9869,0.9935,0.9906,0.9841



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Random Forest achieves 99.35% accuracy** and a weighted F1-score of 0.9935, demonstrating strong performance with Word TF-IDF features.
- **Macro F1-score reaches 0.9869**, outperforming Logistic Regression (0.9859) and SGD Classifier (0.9823), although remaining below Linear SVM (0.9924).
- **Key Finding:** Random Forest provides competitive nonlinear classification performance, while Linear SVM remains the strongest overall model.

#### Evaluating Extra Trees (Word TF-IDF)

In [13]:
et_model = ExtraTreesClassifier(
    n_estimators=150,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=1
)

et_cv_results = cross_validate(
    et_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=2
)

et_metrics = {
    metric: et_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Extra Trees - Word TF-IDF"):
    mlflow.log_params({
        "model": "ExtraTreesClassifier",
        "n_estimators": 150,
        "max_features": "sqrt",
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in et_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Extra Trees",
    "Feature Representation": "Word TF-IDF",
    **et_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719
6,Random Forest,Word TF-IDF,0.9935,0.9869,0.9935,0.9906,0.9841
7,Extra Trees,Word TF-IDF,0.9950,0.9869,0.9950,0.9901,0.9852



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Extra Trees achieves 99.50% accuracy** and a weighted F1-score of 0.9950, slightly outperforming Random Forest on both metrics.
- **Macro F1-score reaches 0.9869**, matching Random Forest, while macro recall improves marginally from 0.9841 to 0.9852.
- **Key Finding:** Extra Trees is a competitive tree-based classifier, although Linear SVM remains the strongest overall model with a macro F1-score of 0.9924.

#### Evaluating LightGBM (Word TF-IDF)

In [14]:
lgbm_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=150,
    learning_rate=0.1,
    num_leaves=15,
    max_depth=7,
    min_child_samples=20,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    verbosity=-1,
    random_state=42,
    n_jobs=2
)

lgbm_cv_results = cross_validate(
    lgbm_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

lgbm_metrics = {
    metric: lgbm_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="LightGBM - Word TF-IDF"):
    mlflow.log_params({
        "model": "LGBMClassifier",
        "n_estimators": 150,
        "learning_rate": 0.1,
        "num_leaves": 15,
        "max_depth": 7,
        "min_child_samples": 20,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in lgbm_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "LightGBM",
    "Feature Representation": "Word TF-IDF",
    **lgbm_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719
6,Random Forest,Word TF-IDF,0.9935,0.9869,0.9935,0.9906,0.9841
7,Extra Trees,Word TF-IDF,0.9950,0.9869,0.9950,0.9901,0.9852
8,LightGBM,Word TF-IDF,0.9913,0.9761,0.9912,0.9815,0.9724



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **LightGBM achieves 99.13% accuracy** and a weighted F1-score of 0.9912, demonstrating strong multiclass classification performance.
- **Macro F1-score reaches 0.9761**, falling below Random Forest (0.9869), Extra Trees (0.9869), and Linear SVM (0.9924).
- **Key Finding:** Gradient boosting performs effectively on Word TF-IDF features, although Linear SVM remains the strongest overall classifier.

## 8. Neural Network Model Development

#### Evaluating Multi-Layer Perceptron (Word TF-IDF)

In [15]:
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)

mlp_cv_results = cross_validate(
    mlp_model,
    X_train_word,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

mlp_metrics = {
    metric: mlp_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="MLP Classifier - Word TF-IDF"):
    mlflow.log_params({
        "model": "MLPClassifier",
        "hidden_layer_sizes": "(128,)",
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.0001,
        "batch_size": 128,
        "max_iter": 50,
        "early_stopping": True,
        "feature_representation": "word_tfidf",
        "cv_folds": 5
    })

    for metric, value in mlp_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "MLP Classifier",
    "Feature Representation": "Word TF-IDF",
    **mlp_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(results_df.round(4))

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
0,Dummy Classifier,Word TF-IDF,0.2029,0.0125,0.0685,0.0075,0.0370
1,Multinomial Naive Bayes,Word TF-IDF,0.9794,0.8921,0.9767,0.9511,0.8699
2,Complement Naive Bayes,Word TF-IDF,0.9813,0.9339,0.9807,0.9727,0.9177
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
5,SGD Classifier,Word TF-IDF,0.9944,0.9823,0.9943,0.9954,0.9719
6,Random Forest,Word TF-IDF,0.9935,0.9869,0.9935,0.9906,0.9841
7,Extra Trees,Word TF-IDF,0.9950,0.9869,0.9950,0.9901,0.9852
8,LightGBM,Word TF-IDF,0.9913,0.9761,0.9912,0.9815,0.9724
9,MLP Classifier,Word TF-IDF,0.9966,0.9892,0.9965,0.9948,0.9851



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **MLP Classifier achieves 99.66% accuracy** and a weighted F1-score of 0.9965, demonstrating excellent performance with Word TF-IDF features.
- **Macro F1-score reaches 0.9892**, outperforming Logistic Regression, Random Forest, and Extra Trees, while remaining slightly below Linear SVM (0.9924).
- **Key Finding:** MLP Classifier is the second-best model evaluated so far, highlighting the effectiveness of a lightweight neural network for customer intent classification.

## 9. Feature Representation Comparison

#### Evaluating Linear SVM (Character TF-IDF)

In [16]:
svm_char_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

svm_char_cv_results = cross_validate(
    svm_char_model,
    X_train_char,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

svm_char_metrics = {
    metric: svm_char_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Linear SVM - Character TF-IDF"):
    mlflow.log_params({
        "model": "LinearSVC",
        "C": 1.0,
        "max_iter": 2000,
        "feature_representation": "character_tfidf",
        "cv_folds": 5
    })

    for metric, value in svm_char_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Linear SVM",
    "Feature Representation": "Character TF-IDF",
    **svm_char_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "Linear SVM"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
10,Linear SVM,Character TF-IDF,0.9995,0.9978,0.9995,0.9987,0.9971



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Linear SVM with Character TF-IDF achieves 99.95% accuracy** and a macro F1-score of 0.9978, outperforming the Word TF-IDF representation.
- **Macro recall improves from 0.9904 to 0.9971**, indicating stronger classification performance across the 27 intent classes.
- **Key Finding:** Character-level n-grams provide highly effective predictive features, making Character TF-IDF the strongest representation evaluated so far.

#### Evaluating Linear SVM (Combined TF-IDF)

In [17]:
svm_combined_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

svm_combined_cv_results = cross_validate(
    svm_combined_model,
    X_train_combined,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

svm_combined_metrics = {
    metric: svm_combined_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Linear SVM - Combined TF-IDF"):
    mlflow.log_params({
        "model": "LinearSVC",
        "C": 1.0,
        "max_iter": 2000,
        "feature_representation": "combined_tfidf",
        "cv_folds": 5
    })

    for metric, value in svm_combined_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Linear SVM",
    "Feature Representation": "Combined TF-IDF",
    **svm_combined_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "Linear SVM"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
4,Linear SVM,Word TF-IDF,0.9975,0.9924,0.9975,0.9954,0.9904
10,Linear SVM,Character TF-IDF,0.9995,0.9978,0.9995,0.9987,0.9971
11,Linear SVM,Combined TF-IDF,0.9995,0.9980,0.9995,0.9989,0.9974



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Combined TF-IDF achieves 99.95% accuracy** and a macro F1-score of 0.9980, delivering the strongest Linear SVM performance.
- **Character TF-IDF performs nearly identically**, with a macro F1-score of 0.9978, while Word TF-IDF achieves 0.9924.
- **Key Finding:** Combining word-level and character-level features provides a marginal improvement, making Combined TF-IDF the leading representation for Linear SVM.

#### Evaluating Logistic Regression (Character TF-IDF)

In [18]:
lr_char_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

lr_char_cv_results = cross_validate(
    lr_char_model,
    X_train_char,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

lr_char_metrics = {
    metric: lr_char_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Logistic Regression - Character TF-IDF"):
    mlflow.log_params({
        "model": "LogisticRegression",
        "C": 1.0,
        "max_iter": 1000,
        "solver": "lbfgs",
        "feature_representation": "character_tfidf",
        "cv_folds": 5
    })

    for metric, value in lr_char_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Logistic Regression",
    "Feature Representation": "Character TF-IDF",
    **lr_char_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "Logistic Regression"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
12,Logistic Regression,Character TF-IDF,0.9985,0.9950,0.9985,0.9976,0.9930



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Logistic Regression with Character TF-IDF achieves 99.85% accuracy** and a macro F1-score of 0.9950, outperforming its Word TF-IDF counterpart.
- **Macro recall improves from 0.9778 to 0.9930**, indicating better performance across the 27 intent classes.
- **Key Finding:** Character-level features substantially improve Logistic Regression, reinforcing their effectiveness for customer intent classification.

#### Evaluating Logistic Regression (Combined TF-IDF)

In [19]:
lr_combined_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    random_state=42
)

lr_combined_cv_results = cross_validate(
    lr_combined_model,
    X_train_combined,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

lr_combined_metrics = {
    metric: lr_combined_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="Logistic Regression - Combined TF-IDF"):
    mlflow.log_params({
        "model": "LogisticRegression",
        "C": 1.0,
        "max_iter": 1000,
        "solver": "lbfgs",
        "feature_representation": "combined_tfidf",
        "cv_folds": 5
    })

    for metric, value in lr_combined_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "Logistic Regression",
    "Feature Representation": "Combined TF-IDF",
    **lr_combined_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "Logistic Regression"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
3,Logistic Regression,Word TF-IDF,0.9957,0.9859,0.9956,0.9960,0.9778
12,Logistic Regression,Character TF-IDF,0.9985,0.9950,0.9985,0.9976,0.9930
13,Logistic Regression,Combined TF-IDF,0.9987,0.9939,0.9987,0.9976,0.9911



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Combined TF-IDF achieves the highest accuracy of 99.87%**, marginally outperforming Character TF-IDF (99.85%) and Word TF-IDF (99.57%).
- **Character TF-IDF achieves the highest macro F1-score of 0.9950** and macro recall of 0.9930, demonstrating stronger balanced performance across intent classes.
- **Key Finding:** Character TF-IDF is the preferred representation for Logistic Regression based on macro F1-score, while Combined TF-IDF offers slightly higher overall accuracy.

#### Evaluating MLP Classifier (Character TF-IDF)

In [20]:
mlp_char_model = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)

mlp_char_cv_results = cross_validate(
    mlp_char_model,
    X_train_char,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

mlp_char_metrics = {
    metric: mlp_char_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="MLP Classifier - Character TF-IDF"):
    mlflow.log_params({
        "model": "MLPClassifier",
        "hidden_layer_sizes": "(128,)",
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.0001,
        "batch_size": 128,
        "max_iter": 50,
        "early_stopping": True,
        "feature_representation": "character_tfidf",
        "cv_folds": 5
    })

    for metric, value in mlp_char_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "MLP Classifier",
    "Feature Representation": "Character TF-IDF",
    **mlp_char_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "MLP Classifier"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
9,MLP Classifier,Word TF-IDF,0.9966,0.9892,0.9965,0.9948,0.9851
14,MLP Classifier,Character TF-IDF,0.9983,0.9952,0.9982,0.9972,0.9938



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **MLP Classifier with Character TF-IDF achieves 99.83% accuracy** and a weighted F1-score of 0.9982, outperforming its Word TF-IDF counterpart.
- **Macro F1-score improves from 0.9892 to 0.9952**, while macro recall increases from 0.9851 to 0.9938.
- **Key Finding:** Character-level features substantially improve MLP performance, making Character TF-IDF the preferred representation evaluated so far.

#### Evaluating MLP Classifier (Combined TF-IDF)

In [21]:
mlp_combined_model = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=128,
    learning_rate_init=0.001,
    max_iter=50,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)

mlp_combined_cv_results = cross_validate(
    mlp_combined_model,
    X_train_combined,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1
)

mlp_combined_metrics = {
    metric: mlp_combined_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

with mlflow.start_run(run_name="MLP Classifier - Combined TF-IDF"):
    mlflow.log_params({
        "model": "MLPClassifier",
        "hidden_layer_sizes": "(128,)",
        "activation": "relu",
        "solver": "adam",
        "alpha": 0.0001,
        "batch_size": 128,
        "max_iter": 50,
        "early_stopping": True,
        "feature_representation": "combined_tfidf",
        "cv_folds": 5
    })

    for metric, value in mlp_combined_metrics.items():
        mlflow.log_metric(f"cv_{metric}", value)

model_results.append({
    "Model": "MLP Classifier",
    "Feature Representation": "Combined TF-IDF",
    **mlp_combined_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

display(
    results_df[
        results_df["Model"] == "MLP Classifier"
    ].round(4)
)

print(f"\nResults saved to: {results_path}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted,precision_macro,recall_macro
9,MLP Classifier,Word TF-IDF,0.9966,0.9892,0.9965,0.9948,0.9851
14,MLP Classifier,Character TF-IDF,0.9983,0.9952,0.9982,0.9972,0.9938
15,MLP Classifier,Combined TF-IDF,0.9992,0.9959,0.9992,0.9982,0.9942



Results saved to: ..\data\processed\model_comparison_results.csv


#### Observations
- **Combined TF-IDF achieves the highest accuracy of 99.92%** and macro F1-score of 0.9959, outperforming both Word and Character TF-IDF.
- **Macro recall improves from 0.9851 to 0.9942** compared with Word TF-IDF, indicating stronger performance across the 27 intent classes.
- **Key Finding:** Combining word-level and character-level features provides the strongest MLP performance among the evaluated representations.

## 10. Hyperparameter Tuning

#### Tuning Linear SVM (Combined TF-IDF)

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV

svm_tuning_pipeline = Pipeline([
    ("features", ColumnTransformer([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=True,
            strip_accents="unicode",
            dtype=np.float32
        ), "utterance"),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            lowercase=True,
            strip_accents="unicode",
            dtype=np.float32
        ), "utterance")
    ])),
    ("classifier", LinearSVC(
        max_iter=2000,
        random_state=42
    ))
])

svm_param_grid = {
    "classifier__C": [0.1, 0.5, 1.0, 2.0, 5.0]
}

svm_grid_search = GridSearchCV(
    estimator=svm_tuning_pipeline,
    param_grid=svm_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

svm_grid_search.fit(train_df[["utterance"]], y_train)

print("Best Parameters:", svm_grid_search.best_params_)
print("Best CV Macro F1:", round(svm_grid_search.best_score_, 4))

Best Parameters: {'classifier__C': 2.0}
Best CV Macro F1: 0.9968


#### Observations
- **Grid search identifies C=2.0** as the optimal regularization parameter among the five evaluated values.
- **The tuned pipeline achieves a macro F1-score of 0.9968** using five-fold stratified cross-validation.
- **Key Finding:** Leakage-free feature extraction provides a more reliable performance estimate, although the earlier precomputed-feature CV score is not directly comparable.

#### Logging and Saving the Tuned Linear SVM

In [23]:
best_svm_pipeline = svm_grid_search.best_estimator_

svm_tuned_metrics = {
    "f1_macro": svm_grid_search.best_score_
}

with mlflow.start_run(run_name="Linear SVM - Tuned Combined TF-IDF"):
    mlflow.log_params({
        "model": "LinearSVC",
        "C": svm_grid_search.best_params_["classifier__C"],
        "max_iter": 2000,
        "feature_representation": "combined_tfidf",
        "cv_folds": 5,
        "tuning_method": "GridSearchCV",
        "cv_vectorization": "within_fold"
    })

    mlflow.log_metric("cv_f1_macro", svm_grid_search.best_score_)
    mlflow.sklearn.log_model(
        best_svm_pipeline,
        name="tuned_linear_svm_pipeline"
    )

svm_model_path = models_dir / "tuned_linear_svm_pipeline.joblib"
joblib.dump(best_svm_pipeline, svm_model_path)

model_results.append({
    "Model": "Linear SVM (Tuned)",
    "Feature Representation": "Combined TF-IDF",
    "accuracy": np.nan,
    "f1_macro": svm_grid_search.best_score_,
    "f1_weighted": np.nan,
    "precision_macro": np.nan,
    "recall_macro": np.nan
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

print(f"Best C: {svm_grid_search.best_params_['classifier__C']}")
print(f"Best CV Macro F1: {svm_grid_search.best_score_:.4f}")
print(f"Model saved to: {svm_model_path}")
print(f"Results saved to: {results_path}")

Best C: 2.0
Best CV Macro F1: 0.9968
Model saved to: ..\models\tuned_linear_svm_pipeline.joblib
Results saved to: ..\data\processed\model_comparison_results.csv


#### Tuning Logistic Regression (Character TF-IDF)

In [24]:
lr_tuning_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=True,
        strip_accents="unicode",
        dtype=np.float32
    )),
    ("classifier", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    ))
])

lr_param_grid = {
    "classifier__C": [0.1, 0.5, 1.0, 2.0, 5.0]
}

lr_grid_search = GridSearchCV(
    estimator=lr_tuning_pipeline,
    param_grid=lr_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=2,
    refit=True,
    return_train_score=True
)

lr_grid_search.fit(train_df["utterance"], y_train)

print("Best Parameters:", lr_grid_search.best_params_)
print("Best CV Macro F1:", round(lr_grid_search.best_score_, 4))

Best Parameters: {'classifier__C': 5.0}
Best CV Macro F1: 0.9969


#### Observations
- **Grid search identifies C=5.0** as the best Logistic Regression regularization parameter among the five evaluated values.
- **The tuned Character TF-IDF pipeline achieves a macro F1-score of 0.9969**, marginally exceeding the tuned Linear SVM score of 0.9968.
- **Key Finding:** Logistic Regression and Linear SVM demonstrate nearly identical cross-validation performance, making both strong candidates for final evaluation.

#### Logging and Saving the Tuned Logistic Regression

In [25]:
best_lr_pipeline = lr_grid_search.best_estimator_

with mlflow.start_run(run_name="Logistic Regression - Tuned Character TF-IDF"):
    mlflow.log_params({
        "model": "LogisticRegression",
        "C": lr_grid_search.best_params_["classifier__C"],
        "solver": "lbfgs",
        "max_iter": 1000,
        "feature_representation": "character_tfidf",
        "cv_folds": 5,
        "tuning_method": "GridSearchCV",
        "cv_vectorization": "within_fold"
    })

    mlflow.log_metric("cv_f1_macro", lr_grid_search.best_score_)

    mlflow.sklearn.log_model(
        best_lr_pipeline,
        name="tuned_logistic_regression_pipeline"
    )

lr_model_path = models_dir / "tuned_logistic_regression_pipeline.joblib"
joblib.dump(best_lr_pipeline, lr_model_path)

model_results.append({
    "Model": "Logistic Regression (Tuned)",
    "Feature Representation": "Character TF-IDF",
    "accuracy": np.nan,
    "f1_macro": lr_grid_search.best_score_,
    "f1_weighted": np.nan,
    "precision_macro": np.nan,
    "recall_macro": np.nan
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

print(f"Best C: {lr_grid_search.best_params_['classifier__C']}")
print(f"Best CV Macro F1: {lr_grid_search.best_score_:.4f}")
print(f"Model saved to: {lr_model_path}")
print(f"Results saved to: {results_path}")

Best C: 5.0
Best CV Macro F1: 0.9969
Model saved to: ..\models\tuned_logistic_regression_pipeline.joblib
Results saved to: ..\data\processed\model_comparison_results.csv


#### 11. Deep Learning Model Development

In [28]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cpu')}")

PyTorch version: 2.15.0+cpu
Device: cpu


#### Preparing Text Data for Deep Learning

In [29]:
from collections import Counter
import re

train_texts = train_df["utterance"].astype(str).tolist()
test_texts = test_df["utterance"].astype(str).tolist()

def tokenize_text(text):
    return re.findall(r"\b\w+\b|[^\w\s]", text.lower())

token_counts = Counter(
    token
    for text in train_texts
    for token in tokenize_text(text)
)

min_token_frequency = 2

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for token, count in token_counts.items():
    if count >= min_token_frequency:
        vocab[token] = len(vocab)

print(f"Training samples: {len(train_texts):,}")
print(f"Test samples: {len(test_texts):,}")
print(f"Vocabulary size: {len(vocab):,}")
print(f"Unique tokens: {len(token_counts):,}")

Training samples: 17,210
Test samples: 4,303
Vocabulary size: 656
Unique tokens: 1,753


#### Observations
- **The training dataset contains 17,210 utterances**, while 4,303 samples remain reserved for final testing.
- **The vocabulary contains 656 tokens**, including padding and unknown-token placeholders, after applying a minimum frequency threshold of two.
- **Key Decision:** The vocabulary is constructed exclusively from training data, preventing test-set information leakage during token preparation.

#### Analyzing Token Sequence Lengths

In [30]:
sequence_lengths = np.array([
    len(tokenize_text(text))
    for text in train_texts
])

sequence_length_summary = pd.Series(sequence_lengths).describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
)

display(sequence_length_summary.round(2))

print(f"\nMaximum sequence length: {sequence_lengths.max()}")
print(f"Sequences longer than 40 tokens: {(sequence_lengths > 40).sum():,}")
print(f"Percentage within 40 tokens: {(sequence_lengths <= 40).mean() * 100:.2f}%")

count    17210.00
mean        12.28
std          4.26
min          2.00
50%         12.00
90%         17.00
95%         20.00
99%         25.00
max         31.00
dtype: float64


Maximum sequence length: 31
Sequences longer than 40 tokens: 0
Percentage within 40 tokens: 100.00%


#### Observations
- **Utterances contain an average of 12.28 tokens**, with a median sequence length of 12.
- **99% of training utterances contain 25 tokens or fewer**, while the longest contains 31 tokens.
- **Key Decision:** A maximum sequence length of 32 tokens is selected to preserve all training sequences while maintaining computational efficiency.

#### Encoding and Padding Text Sequences

In [31]:
max_sequence_length = 32

def encode_text(text, vocabulary, max_length):
    tokens = tokenize_text(text)

    token_ids = [
        vocabulary.get(token, vocabulary["<UNK>"])
        for token in tokens[:max_length]
    ]

    padding_length = max_length - len(token_ids)
    token_ids.extend([vocabulary["<PAD>"]] * padding_length)

    return token_ids

X_train_sequences = np.array(
    [
        encode_text(text, vocab, max_sequence_length)
        for text in train_texts
    ],
    dtype=np.int64
)

X_test_sequences = np.array(
    [
        encode_text(text, vocab, max_sequence_length)
        for text in test_texts
    ],
    dtype=np.int64
)

print(f"Training sequences shape: {X_train_sequences.shape}")
print(f"Test sequences shape: {X_test_sequences.shape}")
print(f"Sequence length: {max_sequence_length}")
print(f"Vocabulary size: {len(vocab)}")
print(f"Sample encoded sequence:\n{X_train_sequences[0]}")

Training sequences shape: (17210, 32)
Test sequences shape: (4303, 32)
Sequence length: 32
Vocabulary size: 656
Sample encoded sequence:
[2 3 4 5 6 7 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


#### Observations
- **Training and test sequences have shapes (17,210, 32) and (4,303, 32)**, respectively.
- **Each utterance is represented by integer token IDs**, with zero-padding applied to shorter sequences.
- **Key Decision:** The fixed sequence length of 32 provides a consistent input format for both TextCNN and BiLSTM.



#### Encoding Intent Labels

In [32]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)

joblib.dump(label_encoder, models_dir / "intent_label_encoder.joblib")
joblib.dump(vocab, models_dir / "deep_learning_vocabulary.joblib")

np.save(features_dir / "X_train_sequences.npy", X_train_sequences)
np.save(features_dir / "X_test_sequences.npy", X_test_sequences)
np.save(features_dir / "y_train_encoded.npy", y_train_encoded)
np.save(features_dir / "y_test_encoded.npy", y_test_encoded)

print(f"Number of intent classes: {num_classes}")
print(f"Encoded training labels: {y_train_encoded.shape}")
print(f"Encoded test labels: {y_test_encoded.shape}")
print(f"Label range: {y_train_encoded.min()} to {y_train_encoded.max()}")
print("Vocabulary, label encoder, and encoded arrays saved successfully.")

Number of intent classes: 27
Encoded training labels: (17210,)
Encoded test labels: (4303,)
Label range: 0 to 26
Vocabulary, label encoder, and encoded arrays saved successfully.


#### Observations
- **All 27 intent classes are successfully encoded** into numerical labels ranging from 0 to 26.
- **Training and test labels contain 17,210 and 4,303 samples**, respectively, matching their corresponding text sequences.
- **Key Decision:** The vocabulary, label encoder, and encoded arrays are saved for reproducible deep learning experiments and future model evaluation.

#### Creating Stratified Training and Validation Splits

In [33]:
from sklearn.model_selection import train_test_split

X_dl_train, X_dl_val, y_dl_train, y_dl_val = train_test_split(
    X_train_sequences,
    y_train_encoded,
    test_size=0.15,
    stratify=y_train_encoded,
    random_state=42
)

np.save(features_dir / "X_dl_train.npy", X_dl_train)
np.save(features_dir / "X_dl_val.npy", X_dl_val)
np.save(features_dir / "y_dl_train.npy", y_dl_train)
np.save(features_dir / "y_dl_val.npy", y_dl_val)

print(f"Deep learning training samples: {len(X_dl_train):,}")
print(f"Deep learning validation samples: {len(X_dl_val):,}")
print(f"Original test samples (untouched): {len(X_test_sequences):,}")
print(f"Training classes: {len(np.unique(y_dl_train))}")
print(f"Validation classes: {len(np.unique(y_dl_val))}")

Deep learning training samples: 14,628
Deep learning validation samples: 2,582
Original test samples (untouched): 4,303
Training classes: 27
Validation classes: 27


#### Observations
- **The deep learning training set contains 14,628 samples**, while 2,582 samples are reserved for validation.
- **All 27 intent classes are represented in both subsets**, preserving coverage of the multiclass classification problem.
- **Key Decision:** The original 4,303 test samples remain reserved for final model evaluation, preventing test-set influence during model development.

#### Creating PyTorch Datasets and Data Loaders

In [34]:
import torch
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

train_dataset = TensorDataset(
    torch.tensor(X_dl_train, dtype=torch.long),
    torch.tensor(y_dl_train, dtype=torch.long)
)

val_dataset = TensorDataset(
    torch.tensor(X_dl_val, dtype=torch.long),
    torch.tensor(y_dl_val, dtype=torch.long)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

device = torch.device("cpu")

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Batch size: {batch_size}")
print(f"Device: {device}")

sample_inputs, sample_labels = next(iter(train_loader))

print(f"Input batch shape: {sample_inputs.shape}")
print(f"Label batch shape: {sample_labels.shape}")

Training batches: 229
Validation batches: 41
Batch size: 64
Device: cpu
Input batch shape: torch.Size([64, 32])
Label batch shape: torch.Size([64])


#### Observations
- **The training and validation datasets contain 229 and 41 batches**, respectively, using a batch size of 64.
- **Each input batch has shape (64, 32)**, representing 64 utterances with 32 token positions each.
- **Key Decision:** CPU-based mini-batch processing is configured for efficient TextCNN and BiLSTM training.

#### Building the TextCNN Architecture

In [35]:
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes,
        embedding_dim=64,
        num_filters=64,
        kernel_sizes=(3, 4, 5),
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.convolutions = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=kernel_size
            )
            for kernel_size in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

    def forward(self, x):
        embedded = self.embedding(x)
        embedded = embedded.permute(0, 2, 1)

        pooled_features = [
            F.max_pool1d(
                F.relu(conv(embedded)),
                kernel_size=conv(embedded).shape[2]
            ).squeeze(2)
            for conv in self.convolutions
        ]

        combined_features = torch.cat(pooled_features, dim=1)
        combined_features = self.dropout(combined_features)

        return self.classifier(combined_features)


textcnn_model = TextCNN(
    vocab_size=len(vocab),
    num_classes=num_classes
).to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in textcnn_model.parameters()
)

print(textcnn_model)
print(f"\nTotal parameters: {total_parameters:,}")

TextCNN(
  (embedding): Embedding(656, 64, padding_idx=0)
  (convolutions): ModuleList(
    (0): Conv1d(64, 64, kernel_size=(3,), stride=(1,))
    (1): Conv1d(64, 64, kernel_size=(4,), stride=(1,))
    (2): Conv1d(64, 64, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=192, out_features=27, bias=True)
)

Total parameters: 96,539


In [36]:
from sklearn.metrics import f1_score
import copy

torch.manual_seed(42)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    textcnn_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

max_epochs = 15
patience = 3

best_val_f1 = -1.0
best_model_state = None
epochs_without_improvement = 0

training_history = []

for epoch in range(max_epochs):
    textcnn_model.train()
    train_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = textcnn_model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)

    textcnn_model.eval()
    val_predictions = []
    val_targets = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)

            outputs = textcnn_model(inputs)
            predictions = outputs.argmax(dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_targets.extend(labels.numpy())

    val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro",
        zero_division=0
    )

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_f1_macro": val_f1
    })

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(textcnn_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}.")
        break

textcnn_model.load_state_dict(best_model_state)

print(f"\nBest Validation Macro F1: {best_val_f1:.4f}")

Epoch 01 | Train Loss: 0.7995 | Val Macro F1: 0.9036
Epoch 02 | Train Loss: 0.0910 | Val Macro F1: 0.9839
Epoch 03 | Train Loss: 0.0438 | Val Macro F1: 0.9882
Epoch 04 | Train Loss: 0.0299 | Val Macro F1: 0.9959
Epoch 05 | Train Loss: 0.0238 | Val Macro F1: 0.9937
Epoch 06 | Train Loss: 0.0199 | Val Macro F1: 0.9957
Epoch 07 | Train Loss: 0.0180 | Val Macro F1: 0.9953

Early stopping triggered at epoch 7.

Best Validation Macro F1: 0.9959


#### Observations
- **TextCNN achieves a peak validation macro F1-score of 0.9959 at epoch 4**, demonstrating strong performance across the 27 intent classes.
- **Training loss decreases from 0.7995 to 0.0180**, while validation performance stabilizes after the fourth epoch.
- **Key Finding:** Early stopping terminates training at epoch 7 and restores the best-performing model weights, reducing unnecessary training and potential overfitting.

#### Logging and Saving the TextCNN Model

In [37]:
textcnn_model_path = models_dir / "textcnn_model.pth"
textcnn_history_path = processed_dir / "textcnn_training_history.csv"

torch.save(textcnn_model.state_dict(), textcnn_model_path)

textcnn_history_df = pd.DataFrame(training_history)
textcnn_history_df.to_csv(textcnn_history_path, index=False)

with mlflow.start_run(run_name="TextCNN - PyTorch"):
    mlflow.log_params({
        "model": "TextCNN",
        "framework": "PyTorch",
        "embedding_dim": 64,
        "num_filters": 64,
        "kernel_sizes": "3,4,5",
        "dropout": 0.3,
        "batch_size": batch_size,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
        "max_epochs": max_epochs,
        "early_stopping_patience": patience,
        "vocab_size": len(vocab),
        "num_classes": num_classes,
        "feature_representation": "Token Sequences",
        "evaluation_method": "Stratified Validation Split"
    })

    mlflow.log_metric("best_val_f1_macro", best_val_f1)
    mlflow.log_metric("epochs_trained", len(training_history))
    mlflow.log_metric(
        "best_epoch",
        int(textcnn_history_df.loc[
            textcnn_history_df["val_f1_macro"].idxmax(), "epoch"
        ])
    )

    mlflow.log_artifact(str(textcnn_model_path))
    mlflow.log_artifact(str(textcnn_history_path))

model_results.append({
    "Model": "TextCNN",
    "Feature Representation": "Token Sequences",
    "accuracy": np.nan,
    "f1_macro": best_val_f1,
    "f1_weighted": np.nan,
    "precision_macro": np.nan,
    "recall_macro": np.nan
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

print(f"Best Validation Macro F1: {best_val_f1:.4f}")
print(f"Model saved to: {textcnn_model_path}")
print(f"Training history saved to: {textcnn_history_path}")
print(f"Results saved to: {results_path}")

Best Validation Macro F1: 0.9959
Model saved to: ..\models\textcnn_model.pth
Training history saved to: ..\data\processed\textcnn_training_history.csv
Results saved to: ..\data\processed\model_comparison_results.csv


#### Building the BiLSTM Architecture

In [38]:
from torch.nn.utils.rnn import pack_padded_sequence

class BiLSTM(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes,
        embedding_dim=64,
        hidden_dim=64,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x):
        lengths = (x != 0).sum(dim=1).cpu()

        embedded = self.embedding(x)

        packed = pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        combined_hidden = torch.cat(
            (hidden[-2], hidden[-1]),
            dim=1
        )

        combined_hidden = self.dropout(combined_hidden)

        return self.classifier(combined_hidden)


torch.manual_seed(42)

bilstm_model = BiLSTM(
    vocab_size=len(vocab),
    num_classes=num_classes
).to(device)

total_parameters = sum(
    parameter.numel()
    for parameter in bilstm_model.parameters()
)

print(bilstm_model)
print(f"\nTotal parameters: {total_parameters:,}")

BiLSTM(
  (embedding): Embedding(656, 64, padding_idx=0)
  (lstm): LSTM(64, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=128, out_features=27, bias=True)
)

Total parameters: 112,027


#### Observations
- **BiLSTM contains 112,027 trainable parameters**, maintaining a lightweight architecture suitable for CPU-based training.
- **The bidirectional LSTM uses 64 hidden units per direction**, producing a combined 128-dimensional representation for classification.
- **Key Decision:** Packed sequences are used to exclude padding tokens from recurrent processing, while 30% dropout provides regularization.

#### Training the BiLSTM Model

In [39]:
torch.manual_seed(42)

bilstm_criterion = nn.CrossEntropyLoss()

bilstm_optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

bilstm_max_epochs = 15
bilstm_patience = 3

best_bilstm_val_f1 = -1.0
best_bilstm_state = None
bilstm_epochs_without_improvement = 0

bilstm_training_history = []

for epoch in range(bilstm_max_epochs):
    bilstm_model.train()
    train_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        bilstm_optimizer.zero_grad()

        outputs = bilstm_model(inputs)
        loss = bilstm_criterion(outputs, labels)

        loss.backward()
        bilstm_optimizer.step()

        train_loss += loss.item() * inputs.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)

    bilstm_model.eval()
    val_predictions = []
    val_targets = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)

            outputs = bilstm_model(inputs)
            predictions = outputs.argmax(dim=1)

            val_predictions.extend(predictions.cpu().numpy())
            val_targets.extend(labels.numpy())

    val_f1 = f1_score(
        val_targets,
        val_predictions,
        average="macro",
        zero_division=0
    )

    bilstm_training_history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_f1_macro": val_f1
    })

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    if val_f1 > best_bilstm_val_f1:
        best_bilstm_val_f1 = val_f1
        best_bilstm_state = copy.deepcopy(bilstm_model.state_dict())
        bilstm_epochs_without_improvement = 0
    else:
        bilstm_epochs_without_improvement += 1

    if bilstm_epochs_without_improvement >= bilstm_patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}.")
        break

bilstm_model.load_state_dict(best_bilstm_state)

print(f"\nBest Validation Macro F1: {best_bilstm_val_f1:.4f}")

Epoch 01 | Train Loss: 1.2294 | Val Macro F1: 0.7901
Epoch 02 | Train Loss: 0.1194 | Val Macro F1: 0.9543
Epoch 03 | Train Loss: 0.0452 | Val Macro F1: 0.9858
Epoch 04 | Train Loss: 0.0281 | Val Macro F1: 0.9890
Epoch 05 | Train Loss: 0.0224 | Val Macro F1: 0.9825
Epoch 06 | Train Loss: 0.0141 | Val Macro F1: 0.9869
Epoch 07 | Train Loss: 0.0162 | Val Macro F1: 0.9812

Early stopping triggered at epoch 7.

Best Validation Macro F1: 0.9890


#### Observations
- **BiLSTM achieves a peak validation macro F1-score of 0.9890 at epoch 4**, demonstrating strong multiclass classification performance.
- **Training loss decreases from 1.2294 to 0.0162**, while validation performance fluctuates after epoch 4, triggering early stopping at epoch 7.
- **Key Finding:** TextCNN outperforms BiLSTM in validation macro F1-score (0.9959 vs. 0.9890) while using fewer trainable parameters.

#### Logging and Saving the BiLSTM Model

In [40]:
bilstm_model_path = models_dir / "bilstm_model.pth"
bilstm_history_path = processed_dir / "bilstm_training_history.csv"

torch.save(bilstm_model.state_dict(), bilstm_model_path)

bilstm_history_df = pd.DataFrame(bilstm_training_history)
bilstm_history_df.to_csv(bilstm_history_path, index=False)

with mlflow.start_run(run_name="BiLSTM - PyTorch"):
    mlflow.log_params({
        "model": "BiLSTM",
        "framework": "PyTorch",
        "embedding_dim": 64,
        "hidden_dim": 64,
        "bidirectional": True,
        "dropout": 0.3,
        "batch_size": batch_size,
        "learning_rate": 0.001,
        "weight_decay": 1e-4,
        "max_epochs": bilstm_max_epochs,
        "early_stopping_patience": bilstm_patience,
        "vocab_size": len(vocab),
        "num_classes": num_classes,
        "feature_representation": "Token Sequences",
        "evaluation_method": "Stratified Validation Split"
    })

    mlflow.log_metric("best_val_f1_macro", best_bilstm_val_f1)
    mlflow.log_metric("epochs_trained", len(bilstm_training_history))
    mlflow.log_metric(
        "best_epoch",
        int(bilstm_history_df.loc[
            bilstm_history_df["val_f1_macro"].idxmax(), "epoch"
        ])
    )

    mlflow.log_artifact(str(bilstm_model_path))
    mlflow.log_artifact(str(bilstm_history_path))

model_results.append({
    "Model": "BiLSTM",
    "Feature Representation": "Token Sequences",
    "accuracy": np.nan,
    "f1_macro": best_bilstm_val_f1,
    "f1_weighted": np.nan,
    "precision_macro": np.nan,
    "recall_macro": np.nan
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

print(f"Best Validation Macro F1: {best_bilstm_val_f1:.4f}")
print(f"Model saved to: {bilstm_model_path}")
print(f"Training history saved to: {bilstm_history_path}")
print(f"Results saved to: {results_path}")

Best Validation Macro F1: 0.9890
Model saved to: ..\models\bilstm_model.pth
Training history saved to: ..\data\processed\bilstm_training_history.csv
Results saved to: ..\data\processed\model_comparison_results.csv


## 12. Ensemble Learning

#### Configuring the Soft Voting Ensemble

In [41]:
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone

voting_lr = clone(best_lr_pipeline)

voting_svm = clone(best_svm_pipeline)
voting_svm.set_params(
    classifier=CalibratedClassifierCV(
        estimator=LinearSVC(
            C=2.0,
            max_iter=2000,
            random_state=42
        ),
        cv=3,
        method="sigmoid"
    )
)

soft_voting_model = VotingClassifier(
    estimators=[
        ("logistic_regression", voting_lr),
        ("linear_svm", voting_svm)
    ],
    voting="soft",
    n_jobs=1
)

print("Soft Voting ensemble configured successfully.")
print("Base estimators: Logistic Regression and Calibrated Linear SVM")

Soft Voting ensemble configured successfully.
Base estimators: Logistic Regression and Calibrated Linear SVM


#### Evaluating Soft Voting Ensemble (Combined Classical Models)

In [43]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import clone

voting_lr = Pipeline([
    ("select_text", FunctionTransformer(
        lambda X: X["utterance"],
        validate=False
    )),
    ("tfidf", clone(best_lr_pipeline.named_steps["tfidf"])),
    ("classifier", clone(best_lr_pipeline.named_steps["classifier"]))
])

voting_svm = clone(best_svm_pipeline)

voting_svm.set_params(
    classifier=CalibratedClassifierCV(
        estimator=LinearSVC(
            C=2.0,
            max_iter=2000,
            random_state=42
        ),
        cv=3,
        method="sigmoid"
    )
)

soft_voting_model = VotingClassifier(
    estimators=[
        ("logistic_regression", voting_lr),
        ("linear_svm", voting_svm)
    ],
    voting="soft",
    n_jobs=1
)

soft_voting_cv_results = cross_validate(
    soft_voting_model,
    train_df[["utterance"]],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise"
)

soft_voting_metrics = {
    metric: soft_voting_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

print("Soft Voting Ensemble — Cross-Validation Results")

for metric, value in soft_voting_metrics.items():
    print(f"{metric}: {value:.4f}")

Soft Voting Ensemble — Cross-Validation Results
accuracy: 0.9994
f1_macro: 0.9969
f1_weighted: 0.9994
precision_macro: 0.9984
recall_macro: 0.9961


#### Observations
- **Soft Voting achieves 99.94% cross-validation accuracy and a macro F1-score of 0.9969**, demonstrating strong performance across the 27 intent classes.
- **Macro precision reaches 0.9984, while macro recall is 0.9961**, indicating consistently high classification performance.
- **Key Finding:** Soft Voting matches the tuned Logistic Regression macro F1-score without providing a measurable improvement, suggesting that the simpler individual model remains competitive.

#### Logging Soft Voting Ensemble Results

In [44]:
with mlflow.start_run(run_name="Soft Voting Ensemble"):
    mlflow.log_params({
        "model": "Soft Voting",
        "base_estimators": "Logistic Regression, Calibrated Linear SVM",
        "voting": "soft",
        "svm_calibration": "sigmoid",
        "calibration_cv": 3,
        "cross_validation_folds": 5,
        "feature_representation": "Character + Combined TF-IDF"
    })

    mlflow.log_metrics(soft_voting_metrics)

model_results.append({
    "Model": "Soft Voting Ensemble",
    "Feature Representation": "Character + Combined TF-IDF",
    **soft_voting_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

print("Soft Voting results logged successfully.")
print(f"Results saved to: {results_path}")
print(f"Total model experiments recorded: {len(results_df)}")

Soft Voting results logged successfully.
Results saved to: ..\data\processed\model_comparison_results.csv
Total model experiments recorded: 21


#### Configuring the Stacking Ensemble

In [45]:
from sklearn.ensemble import StackingClassifier

stacking_model = StackingClassifier(
    estimators=[
        ("logistic_regression", clone(voting_lr)),
        ("linear_svm", clone(best_svm_pipeline))
    ],
    final_estimator=LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    cv=3,
    stack_method="auto",
    passthrough=False,
    n_jobs=1
)

print("Stacking ensemble configured successfully.")
print("Base estimators: Logistic Regression and Linear SVM")
print("Meta-classifier: Logistic Regression")
print("Internal stacking folds: 3")

Stacking ensemble configured successfully.
Base estimators: Logistic Regression and Linear SVM
Meta-classifier: Logistic Regression
Internal stacking folds: 3


#### Evaluating the Stacking Ensemble

In [46]:
stacking_cv_results = cross_validate(
    stacking_model,
    train_df[["utterance"]],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    error_score="raise"
)

stacking_metrics = {
    metric: stacking_cv_results[f"test_{metric}"].mean()
    for metric in scoring
}

print("Stacking Ensemble — Cross-Validation Results")

for metric, value in stacking_metrics.items():
    print(f"{metric}: {value:.4f}")

Stacking Ensemble — Cross-Validation Results
accuracy: 0.9996
f1_macro: 0.9971
f1_weighted: 0.9996
precision_macro: 0.9986
recall_macro: 0.9965


#### Observations
- **Stacking achieves 99.96% cross-validation accuracy and a macro F1-score of 0.9971**, delivering the strongest leakage-free cross-validation performance among the evaluated classical models.
- **Macro precision reaches 0.9986 and macro recall reaches 0.9965**, indicating strong classification performance across the 27 intent classes.
- **Key Finding:** Stacking slightly outperforms Soft Voting and the tuned individual models, although the improvement is marginal relative to its additional computational complexity.

#### Logging Stacking Ensemble Results

In [47]:
with mlflow.start_run(run_name="Stacking Ensemble"):
    mlflow.log_params({
        "model": "Stacking",
        "base_estimators": "Logistic Regression, Linear SVM",
        "meta_classifier": "Logistic Regression",
        "internal_cv_folds": 3,
        "external_cv_folds": 5,
        "stack_method": "auto",
        "passthrough": False,
        "feature_representation": "Character + Combined TF-IDF"
    })

    mlflow.log_metrics(stacking_metrics)

model_results.append({
    "Model": "Stacking Ensemble",
    "Feature Representation": "Character + Combined TF-IDF",
    **stacking_metrics
})

results_df = pd.DataFrame(model_results)
results_df.to_csv(results_path, index=False)

print("Stacking results logged successfully.")
print(f"Results saved to: {results_path}")
print(f"Total model experiments recorded: {len(results_df)}")

Stacking results logged successfully.
Results saved to: ..\data\processed\model_comparison_results.csv
Total model experiments recorded: 22


## 13. Model Comparison and Selection

#### Reviewing Model Experiment Results

In [48]:
results_df = pd.read_csv(results_path)

comparison_df = (
    results_df
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
)

display(
    comparison_df[
        ["Model", "Feature Representation", "accuracy", "f1_macro", "f1_weighted"]
    ].round(4)
)

print(f"Total experiments: {len(comparison_df)}")

,Model,Feature Representation,accuracy,f1_macro,f1_weighted
0,Linear SVM,Combined TF-IDF,0.9995,0.9980,0.9995
1,Linear SVM,Character TF-IDF,0.9995,0.9978,0.9995
2,Stacking Ensemble,Character + Combined TF-IDF,0.9996,0.9971,0.9996
3,Logistic Regression (Tuned),Character TF-IDF,NaN,0.9969,NaN
4,Soft Voting Ensemble,Character + Combined TF-IDF,0.9994,0.9969,0.9994
5,Linear SVM (Tuned),Combined TF-IDF,NaN,0.9968,NaN
6,MLP Classifier,Combined TF-IDF,0.9992,0.9959,0.9992
7,TextCNN,Token Sequences,NaN,0.9959,NaN
8,MLP Classifier,Character TF-IDF,0.9983,0.9952,0.9982
9,Logistic Regression,Character TF-IDF,0.9985,0.9950,0.9985


Total experiments: 22


#### Fitting and Saving the Final Tuned Linear Models

In [49]:
from sklearn.base import clone
import joblib

final_svm_model = clone(best_svm_pipeline)
final_lr_model = clone(best_lr_pipeline)

final_svm_model.fit(
    train_df[["utterance"]],
    y_train
)

final_lr_model.fit(
    train_df["utterance"],
    y_train
)

final_svm_path = models_dir / "final_linear_svm_pipeline.joblib"
final_lr_path = models_dir / "final_logistic_regression_pipeline.joblib"

joblib.dump(final_svm_model, final_svm_path)
joblib.dump(final_lr_model, final_lr_path)

print("Final tuned Linear SVM fitted and saved successfully.")
print(f"Path: {final_svm_path}")

print("\nFinal tuned Logistic Regression fitted and saved successfully.")
print(f"Path: {final_lr_path}")

print(f"\nTraining samples used: {len(y_train):,}")

Final tuned Linear SVM fitted and saved successfully.
Path: ..\models\final_linear_svm_pipeline.joblib

Final tuned Logistic Regression fitted and saved successfully.
Path: ..\models\final_logistic_regression_pipeline.joblib

Training samples used: 17,210


#### Fitting and Saving the Final Stacking Ensemble

In [51]:
def select_utterance(X):
    return X["utterance"]

final_stacking_model.named_estimators_[
    "logistic_regression"
].named_steps["select_text"].func = select_utterance

final_stacking_model.estimators[0][1].named_steps[
    "select_text"
].func = select_utterance

final_stacking_path = models_dir / "final_stacking_pipeline.joblib"

joblib.dump(final_stacking_model, final_stacking_path)

print("Final Stacking Ensemble saved successfully.")
print(f"Path: {final_stacking_path}")
print(f"Training samples used: {len(y_train):,}")
print(f"Number of base estimators: {len(final_stacking_model.estimators_)}")
print(f"Number of intent classes: {len(final_stacking_model.classes_)}")

Final Stacking Ensemble saved successfully.
Path: ..\models\final_stacking_pipeline.joblib
Training samples used: 17,210
Number of base estimators: 2
Number of intent classes: 27


#### Verifying Saved Model Artifacts

In [52]:
model_artifacts = {
    "Linear SVM": models_dir / "final_linear_svm_pipeline.joblib",
    "Logistic Regression": models_dir / "final_logistic_regression_pipeline.joblib",
    "Stacking Ensemble": models_dir / "final_stacking_pipeline.joblib",
    "TextCNN": models_dir / "textcnn_model.pth",
    "BiLSTM": models_dir / "bilstm_model.pth"
}

for model_name, artifact_path in model_artifacts.items():
    exists = artifact_path.is_file()
    size_mb = artifact_path.stat().st_size / (1024 ** 2) if exists else 0

    print(
        f"{model_name:<22} | "
        f"{'Saved' if exists else 'Missing':<7} | "
        f"{size_mb:.2f} MB"
    )

Linear SVM             | Saved   | 1.98 MB
Logistic Regression    | Saved   | 0.72 MB
Stacking Ensemble      | Saved   | 2.71 MB
TextCNN                | Saved   | 0.37 MB
BiLSTM                 | Saved   | 0.43 MB


#### Ensuring Stacking Model Portability

In [53]:
from operator import itemgetter

# Replacing the notebook-defined function in both pipeline copies
final_stacking_model.estimators[0][1].named_steps[
    "select_text"
].func = itemgetter("utterance")

final_stacking_model.named_estimators_[
    "logistic_regression"
].named_steps["select_text"].func = itemgetter("utterance")

# Saving the updated ensemble
joblib.dump(final_stacking_model, final_stacking_path)

# Testing loading and prediction
reloaded_stacking_model = joblib.load(final_stacking_path)

sample_texts = pd.DataFrame({
    "utterance": [
        "I want to cancel my order",
        "Where is my package?",
        "How can I reset my password?"
    ]
})

predictions = reloaded_stacking_model.predict(sample_texts)

print("Stacking Ensemble reloaded successfully.\n")

for text, prediction in zip(sample_texts["utterance"], predictions):
    print(f"Text: {text}")
    print(f"Predicted Intent: {prediction}\n")

print(f"Model saved to: {final_stacking_path}")

Stacking Ensemble reloaded successfully.

Text: I want to cancel my order
Predicted Intent: cancel_order

Text: Where is my package?
Predicted Intent: track_order

Text: How can I reset my password?
Predicted Intent: recover_password

Model saved to: ..\models\final_stacking_pipeline.joblib


## 14. Model Development Summary

The model development phase evaluated 22 experiments across traditional machine learning, neural networks, feature representations, hyperparameter tuning, and ensemble learning.

- **Classical Models:** Linear SVM, Logistic Regression, and Stacking Ensemble emerged as the strongest candidates, with leakage-free cross-validation macro F1-scores of 0.9968, 0.9969, and 0.9971, respectively.
- **Deep Learning Models:** TextCNN achieved a validation macro F1-score of 0.9959, outperforming BiLSTM (0.9890).
- **Ensemble Learning:** Stacking delivered the highest leakage-free cross-validation macro F1-score, although its improvement over simpler models was marginal.
- **Model Artifacts:** The final Linear SVM, Logistic Regression, and Stacking pipelines were fitted on all 17,210 training samples and saved. TextCNN and BiLSTM weights were also preserved.
- **Key Decision:** Stacking Ensemble, tuned Logistic Regression, tuned Linear SVM, and TextCNN were shortlisted for final evaluation. The 4,303 held-out test samples remain untouched.

The next phase will assess generalization performance, class-level classification quality, and error patterns before selecting the final model.